# Data preprocessing

In [18]:
import numpy as np
import pandas as pd
import geopandas as gpd
from datetime import datetime
import matplotlib.pyplot as plt
from shapely.geometry import Point

class dataProcess:
    def __init__(self, polygonAddress = None, crs = None):
        if polygonAddress != None:
            self.polygon = gpd.read_file(polygonAddress).set_crs(crs)
        self.crimeMap = None
        
    def dataImport(self, address, summary = False):
        # Parse the crime record .csv file from a path
        # Input:
        #    summary (boolen): show a descriptive summary of the data
        #    selectType: a list of selected crime types
        # Output:
        #    parse and organized dataframe  
        df = pd.read_csv(address)
        TimeOccurTemp = df['TimeOccur']
        TimeOccur = []
        format = '%m/%d/%Y %H:%M:%S'
        for time in TimeOccurTemp:
            t = datetime.strptime(str(time), format)
            TimeOccur.append(t)
        df.loc[:,('TimeOccur')] = TimeOccur
        dff = df.drop(columns=['TimeOccur','GeoX','GeoY'])
        df = df.set_index('TimeOccur')
        df.insert(df.shape[1], 'CrimeNumber', list(np.zeros((df.shape[0],), dtype=int))) # Add crime amount column

        if summary == True:
            print(dff.apply(pd.value_counts))
            dff.apply(pd.value_counts).plot(kind='bar')
            plt.show()

        return df

    def crimeRaster(self, df):
        # Count the number of crimes in each polygon
        crimeRaster = np.zeros([1, self.polygon.shape[0]])   
        Xlist = df['GeoX'].tolist()
        Ylist = df['GeoY'].tolist()
        for X, Y in zip(Xlist, Ylist):            
            point = Point(X, Y)
            series = self.polygon.contains(point)
            try: 
                loc = series[series == True].index.tolist()[0]
            except:
                #print("WARNING: There are crime points out of polygon boundary.")
                continue      
            crimeRaster[0, loc] += 1 
        return crimeRaster

    def _findEmpty (self, df):
        # Find the location of cells without any crime in past years
        crimeMap = self.crimeRaster(df)
        self.crimeMap = crimeMap
        print('\nThe shape of each crime map is ', crimeMap.shape, '. \n')  
        zeroloc = np.where(crimeMap == 0) 
        return zeroloc

    def dataResampler(self, df, size_resample, deleteNoCrimeCell = False):
        # Cut points into periods and cells
        # Detect cells with no crime and REDO resampling if needed
        
        zeroloc = self._findEmpty(df)
        self.size_resample = size_resample
        if deleteNoCrimeCell == False:
            def CrimeRaster(df):
                data = self.crimeRaster(df)
                data = np.squeeze(data)
                return data  
            dfSampled = df.resample(size_resample, label='right', closed = 'right', origin = 'end').apply(CrimeRaster)
            dfSampled = dfSampled[1:] # remove the first one which might have insufficient data
            print('\nThe shape of each crime map is ', dfSampled.tolist()[0].shape, '. \n') 

        else:
            def CrimeRaster_clean(df, zeroloc = zeroloc):
                data = self.crimeRaster(df)
                data = np.delete(data, zeroloc[1])
                return data
            dfSampled = df.resample(size_resample, label='right', closed = 'right', origin = 'end').apply(CrimeRaster_clean)
            dfSampled = dfSampled[1:] # remove the first one which might have insufficient data
            
            NoCrimeRatio = zeroloc[1].size / self.crimeMap.size
            print(NoCrimeRatio * 100, '% of the cells have no crime record. \n')
            print('The shape of the cleaned crime map is', dfSampled.tolist()[0].shape, '. \n')   
            
        return dfSampled

    def buildDatasets(self, dfSampled, lenSequence, randomIndex = False):
        data = []
        sequence = []

        for i in dfSampled.values:
            sequence.append(i)  # Store raster
            if len(sequence) == lenSequence: # If this sequence is fully filled
                data.append(sequence) # Add sequence to dataset
                sequence = sequence[1: ] # Delete the first crime map

        data = np.asarray(data)
        dataX = data[ :-1]
        
        dataY = dfSampled.tolist()[lenSequence: ]
        dataY = np.asarray(dataY)

        # Training, validation, and test data
        indexes = np.arange(dataX.shape[0])
        if randomIndex == True:
            np.random.shuffle(indexes)
        trainIndex = indexes[: int(0.8 * dataX.shape[0])]
        valIndex = indexes[int(0.8 * dataX.shape[0]): int(0.9 * dataX.shape[0])]
        global testIndex
        testIndex = indexes[int(0.9 * dataX.shape[0]) :]

        trainX = dataX[trainIndex]
        trainY = dataY[trainIndex]
        valX = dataX[valIndex]
        valY = dataY[valIndex]
        testX = dataX[testIndex]
        testY = dataY[testIndex]
        
        trainX = np.moveaxis(trainX, -1, -2)
        valX = np.moveaxis(valX, -1, -2)
        testX = np.moveaxis(testX, -1, -2)
        
        X4Predict = np.moveaxis(data[-1:], -1, -2) #Used for actual prediction
        
        print('\nThe shape of TrianX is', trainX.shape, '. \n')
        print('The shape of TrainY is', trainY.shape, '. \n')
        print('The shape of ValX is', valX.shape, '. \n')
        print('The shape of ValY is', valY.shape, '. \n')
        print('The shape of TestX is', testX.shape, '. \n')
        print('The shape of TestY is', testY.shape, '. \n')
        print('The shape of X4Predict is', X4Predict.shape, '. \n')
        
        return trainX, trainY, valX, valY, testX, testY, X4Predict

# Import data, including crime records and polygon shapefile
dp = dataProcess('./data/ThiessenPolygons.shp', 'EPSG:2240')
df = dp.dataImport('./data/data_2017-.csv')
dfSampled = dp.dataResampler(df, '168h')
lenSequence = 24
trainX, trainY, valX, valY, testX, testY, X4Predict = dp.buildDatasets(dfSampled, lenSequence, randomIndex = False)


The shape of each crime map is  (1, 280) . 


The shape of each crime map is  (280,) . 


The shape of TrianX is (199, 280, 24) . 

The shape of TrainY is (199, 280) . 

The shape of ValX is (25, 280, 24) . 

The shape of ValY is (25, 280) . 

The shape of TestX is (25, 280, 24) . 

The shape of TestY is (25, 280) . 

The shape of X4Predict is (1, 280, 24) . 



# Crime prediction

In [23]:
import math
import plotly.express as px
import plotly.graph_objects as go
from shapely.geometry import Point
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from spektral.layers import GCNConv
from spektral.utils import gcn_filter
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

class crimePrediction():
    def __init__(self):
        self.mapboxKey = None
        self.pf = None
        
    #############################################    
    ################# Core ######################
    def _rescalePredict(self, predict, crimeMapFull):
        if isinstance(predict, np.ndarray):
            return (predict / np.sum(predict)) * np.sum(crimeMapFull)
        elif tf.is_tensor(predict):
            return (predict / tf.reduce_sum(predict)) * np.sum(crimeMapFull)
        else:
            print('WARNING: Input is not numpy array or tensorflow tensor.')
    
    def _klDivergenceElementWise(self, a, b, y_actual):
        # a and b must be tensorflow tensor
        a = keras.backend.clip(a, keras.backend.epsilon(), 1)
        b = keras.backend.clip(b, keras.backend.epsilon(), 1)
        kl = a * tf.math.log(a / b)
        mask = tf.where(y_actual < 1, tf.ones(tf.shape(y_actual)) * self.lowerMultiplier, y_actual)
        return tf.reduce_sum(tf.math.multiply(mask, kl), axis = -1)

    def _customJSDivergence(self, y_actual, y_pred):
        y_actual = y_actual / tf.reduce_sum(y_actual)
        y_pred = y_pred / tf.reduce_sum(y_pred)
        y_m = (y_actual + y_pred) / 2
        return 0.5 * (self._klDivergenceElementWise(y_actual, y_m, y_actual)) + 0.5 * (self._klDivergenceElementWise(y_pred, y_m, y_actual))
    
    def _soft_round(self, x, alpha, eps = 1e-3):
        # x: `tf.Tensor`. Inputs to the rounding function.
        # alpha: Float or `tf.Tensor`. Controls smoothness of the approximation.
        # eps: Float. Threshold below which `soft_round` will return identity.
        # This guards the gradient of tf.where below against NaNs, while maintaining
        # correctness, as for alpha < eps the result is ignored.
#         alpha_bounded = tf.maximum(alpha, eps)
        alpha_bounded = max(alpha, eps)
        m = tf.floor(x) + .5
        r = x - m
        z = tf.tanh(alpha_bounded / 2.) * 2.
        y = m + tf.tanh(alpha_bounded * r) / z
        # For very low alphas, soft_round behaves like identity
        return tf.where(alpha < eps, x, y, name="soft_round")

    def train(self, TrainX, TrainY, ValX, ValY, lap_train = None, lap_val = None, \
              epochs = 50, batch_size = 6, learning_rate = 0.0001, patienceReduceLR = 3, patienceEarlyStopping = 10, \
              alpha_roundApproximation = 7, lowerMultiplier = 0.8):
        
        crimeMapFull = self.getRule0Plan(dayIterval = 7)
        self.lowerMultiplier = lowerMultiplier
        
        inputs_sequence = keras.layers.Input(shape = (TrainX.shape[1], TrainX.shape[2]))
        inputs_lapacian = keras.layers.Input(shape = (TrainX.shape[1], TrainX.shape[1]))
        x = GCNConv(TrainX.shape[2], activation='relu', use_bias = True)([inputs_sequence, inputs_lapacian])
        x = keras.layers.Permute((2, 1))(x)
        x = keras.layers.LSTM(128, activation = 'relu')(x)
        x = keras.layers.Dense(128, activation = 'relu')(x)
        x = keras.layers.Dense(64, activation = 'relu')(x)
        x = keras.layers.Dense(32, activation = 'relu')(x)
        x = keras.layers.Dense(TrainX.shape[1], activation = 'relu')(x)
        x = self._rescalePredict(x, crimeMapFull)
        outputs = self._soft_round(x, alpha = alpha_roundApproximation)

        model = keras.Model(inputs = [inputs_sequence, inputs_lapacian], outputs = outputs)
        model.compile(optimizer = keras.optimizers.Adam(learning_rate = learning_rate), 
                      loss = self._customJSDivergence,)
        model.summary()
        early_stopping = keras.callbacks.EarlyStopping(monitor = "val_loss", patience = patienceEarlyStopping)
        reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor = "val_loss", patience = patienceReduceLR)

        model.fit(
            [TrainX, lap_train],
            TrainY,
            batch_size = batch_size,
            epochs = epochs,
            validation_data = ([ValX, lap_val], ValY),
            callbacks = [early_stopping, reduce_lr],
#             verbose = 0,
        )
        
        return model
         
    def predict(self, testX, testY, dataPointIndex, lap = None):
        self.testWeek = dataPointIndex
        testX_point = testX[dataPointIndex].copy()
        predict = trainedModel([np.expand_dims(testX_point, axis = 0), np.expand_dims(lap, axis = 0)])
        predict = np.squeeze(predict)
        return predict.tolist()    
    
    
    #######################################################
    #################### Visulization #####################
    def polygonUpdate(self, predict):          
        predictY_list = []
        for Bi in predict:
            predictY_list.append(str(int(Bi)))
        self.pf['Prediction'] = predictY_list

    def crimePolygonMap(self):
        geo_df_select = self.pf[self.pf.Prediction != '0']
        fig = px.choropleth_mapbox(geo_df_select, geojson = geo_df_select.geometry,
                                   locations = geo_df_select.index, 
                                   color = "Prediction", 
                                   color_discrete_sequence = px.colors.sequential.Reds,
                                   center = {"lat": 32.606827, "lon": -83.660198},
                                   zoom = 13, opacity = 0.7)
        fig.update_layout(mapbox_style = "light", 
                          mapbox_accesstoken = self.mapboxKey, 
                          margin = {"r":0,"t":0,"l":0,"b":0}, 
                          showlegend = True, 
                          hovermode = False)
        return fig

    def getRawCrimePoints(self, week):
        dt_1 = dfSampled[lenSequence: ][testIndex][[week]].index
        dt_2 = dt_1 - pd.Timedelta(int(self.size_resample.replace('h','')), 'h') # only valid if data was resampled from now to past
        dt_1_str = dt_1.strftime('%m/%d/%Y %hh:%mm:%ss').tolist()[0]
        dt_2_str = dt_2.strftime('%m/%d/%Y %hh:%mm:%ss').tolist()[0]
        time_mask = (df.index <= dt_1_str) & (df.index >= dt_2_str)
        df_selected = df[time_mask]
        return df_selected

    def crimePointMap(self, week):
        df_selected = self.getRawCrimePoints(week)
        crime_points = [Point(xy) for xy in zip(df_selected.GeoX, df_selected.GeoY)]
        df_selected_clean = df_selected.drop(['GeoX', 'GeoY', 'CrimeNumber'], axis=1)
        gdf_selected = gpd.GeoDataFrame(
            df_selected_clean, crs="EPSG:2240", geometry = crime_points).to_crs("EPSG:4326")

        fig = px.scatter_mapbox(gdf_selected, lat=gdf_selected.geometry.y, lon=gdf_selected.geometry.x,
                                color_discrete_sequence=["fuchsia"], 
                                center = {"lat": 32.606827, "lon": -83.660198},
                                zoom = 11.8, opacity = 0.75,  hover_name = "Type")
        fig.update_layout(mapbox_style="light", mapbox_accesstoken = self.mapboxKey,
                         margin = {"r":0,"t":0,"l":0,"b":0}, showlegend = False)
        return fig
    
    def crimePolygonPointMap(self):
        fig1 = self.crimePolygonMap()
        fig2 = self.crimePointMap(self.testWeek)
        # Combine two figs
        fig2.add_trace(fig1.data[0])
        for i, frame in enumerate(fig2.frames):
            fig2.frames[i].data += (fig1.frames[i].data[0],)
        fig2.update_layout(
            autosize = False,
            width = 800,
            height = 600,
        )        
        fig2.show()
        return fig2

    ################################################################
    ################# Evaluation ###################################
    def getRule0Plan(self, dayIterval = 7):
        dfs = df.iloc[0 : int(len(df) / 10)]
        weeks = int ((dfs.index[0] - dfs.index[-1]).days / dayIterval)
        crimeMapFull = dp.crimeRaster(dfs) / weeks
        crimeMapFull = np.rint(crimeMapFull)
        crimeMapFull = crimeMapFull[0]
        return crimeMapFull

    def compareError(self, staticPlan, testX, testY, drawFig = False):
        staticError_rmse = []
        dynamicError_rmse = []
        staticError_mae = []
        dynamicError_mae = []
        staticError_r2 = []
        dynamicError_r2 = []
        
        for week in range(testY.shape[0]):
            trueDist = testY[week].copy()
            predictY = cp.predict(testX, testY, week, lap = lapacian) 
            predictY = np.asarray(predictY)
            predictY = np.rint(predictY)  
    
            dynamicError_rmse.append(mean_squared_error(predictY, trueDist, squared = False))
            staticError_rmse.append(mean_squared_error(staticPlan, trueDist, squared = False))
            dynamicError_mae.append(mean_absolute_error(predictY, trueDist))
            staticError_mae.append(mean_absolute_error(staticPlan, trueDist))
            dynamicError_r2.append(r2_score(predictY, trueDist))
            staticError_r2.append(r2_score(staticPlan, trueDist))            
             
        if drawFig == True:
            fig = go.Figure(data = [go.Scatter(x = list(range(testY.shape[0])), y = staticError_rmse, name = 'Static')])
            fig.add_trace(go.Scatter(x = list(range(testY.shape[0])), y = dynamicError_rmse, name = 'Dynamic'))
            fig.add_hline(y = sum(staticError_rmse) / len(staticError_rmse), 
                          line_dash = "dot", line_color = 'blue',
                          annotation_text = "Mean - Static: " + str(round(sum(staticError_rmse) / len(staticError_rmse), 3)), annotation_font_size = 15,
                          annotation_font_color = "blue", annotation_position="top right")
            fig.add_hline(y = sum(dynamicError_rmse) / len(dynamicError_rmse), 
                      line_dash="dot", line_color = 'red',
                      annotation_text = "Mean - Dynamic: " + str(round(sum(dynamicError_rmse) / len(dynamicError_rmse), 3)), annotation_font_size = 15,
                      annotation_font_color = "red", annotation_position="bottom right")
            fig.show()
        
        return sum(staticError_rmse) / len(staticError_rmse), sum(dynamicError_rmse) / len(dynamicError_rmse), \
            sum(staticError_mae) / len(staticError_mae), sum(dynamicError_mae) / len(dynamicError_mae), \
            sum(staticError_r2) / len(staticError_r2), sum(dynamicError_r2) / len(dynamicError_r2)
            
    def compareJSdivergence(self, staticPlan, testX, testY, drawFig = False):
        staticJSD = []
        dynamicJSD = []
        
        for week in range(testY.shape[0]):
            trueDist = testY[week].copy()
            predictY = cp.predict(testX, testY, week, lap = lapacian) 
            predictY = np.asarray(predictY)
            predictY = np.rint(predictY)
            
            kl = tf.keras.losses.KLDivergence()
            dynamicJs = kl(trueDist/ np.sum(trueDist), predictY / np.sum(predictY)).numpy()
            dynamicJSD.append(dynamicJs)
            staticJs = kl(trueDist/ np.sum(trueDist), staticPlan / np.sum(staticPlan)).numpy()
            staticJSD.append(staticJs)
        
        if drawFig == True:
            fig = go.Figure(data = [go.Scatter(x = list(range(testY.shape[0])), y = staticJSD, name = 'Static')])
            fig.add_trace(go.Scatter(x = list(range(testY.shape[0])), y = dynamicJSD, name = 'Dynamic'))
            fig.update_layout(
                autosize=False,
                width=1000,
                height=600,
                legend=dict(
                    yanchor="top",
                    y=0.99,
                    xanchor="left",
                    x=0.01
                ),
                template = 'seaborn',
                font_size = 18,
            )
            fig.show()    
        
        return sum(staticJSD) / len(staticJSD), sum(dynamicJSD) / len(dynamicJSD)

In [24]:
# Initialization
cp = crimePrediction()
cp.mapboxKey = 'pk.eyJ1IjoicHhpeW9oIiwiYSI6ImNsMGoxa3h1bzA4ZHQzaW41NWd6dm16am0ifQ.QywfLC6Ut-EhSZLt7nirqQ' 
cp.pf = gpd.read_file("./data/ThiessenPolygons.shp").set_crs("EPSG:2240").to_crs("EPSG:4326")
# cp.pf_road = gpd.read_file("./data/MajorRoads.shp").set_crs("EPSG:2240").to_crs("EPSG:4326")
cp.size_resample = dp.size_resample

# get adjMatrix for graph convolution
adjMatrix = np.load('./data/adjMatrix.npy')
lapacian = gcn_filter(adjMatrix, symmetric = True)
lapacian_train = np.tile(lapacian, (trainX.shape[0], 1, 1))
lapacian_val = np.tile(lapacian, (valX.shape[0], 1, 1))

In [ ]:
# # Train
tf.random.set_seed(10)
trainedModel = cp.train(trainX, trainY, valX, valY, lap_train = lapacian_train, lap_val = lapacian_val, \
                        batch_size = 6, epochs = 10000, learning_rate = 2e-05, patienceReduceLR = 5, \
                       alpha_roundApproximation = 5, lowerMultiplier = 0.9) 

# cp.compareError(cp.getRule0Plan(7), testX, testY)
_, dy_js = cp.compareJSdivergence(cp.getRule0Plan(7), testX, testY, drawFig = False)
# print(dy_js)

Model: "functional_1"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 280, 24)]    0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, 280, 280)]   0                                            
__________________________________________________________________________________________________
gcn_conv (GCNConv)              (None, 280, 24)      600         input_1[0][0]                    
                                                                 input_2[0][0]                    
__________________________________________________________________________________________________
permute (Permute)               (None, 24, 280)      0           gcn_conv[0][0]        

34/34 [==============================] - 0s 14ms/step - loss: 0.0657
Epoch 10/10000
34/34 [==============================] - 1s 15ms/step - loss: 0.0624
Epoch 11/10000
34/34 [==============================] - 1s 16ms/step - loss: 0.0584
Epoch 12/10000
34/34 [==============================] - 0s 15ms/step - loss: 0.0531
Epoch 13/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0473
Epoch 14/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0410
Epoch 15/10000
34/34 [==============================] - 1s 15ms/step - loss: 0.0336
Epoch 16/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0292
Epoch 17/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0271
Epoch 18/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0262
Epoch 19/10000
34/34 [==============================] - 1s 15ms/step - loss: 0.0256
Epoch 20/10000
34/34 [==============================] - 1s 15ms/step - loss: 0.0251
Epoch 2

34/34 [==============================] - 1s 15ms/step - loss: 0.0244
Epoch 31/10000
34/34 [==============================] - 0s 15ms/step - loss: 0.0244
Epoch 32/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 33/10000
34/34 [==============================] - 0s 13ms/step - loss: 0.0245
Epoch 34/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0247
Epoch 35/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0246
Epoch 36/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0246
Epoch 37/10000
34/34 [==============================] - 0s 15ms/step - loss: 0.0244
Epoch 38/10000
34/34 [==============================] - 1s 15ms/step - loss: 0.0246
Epoch 39/10000
34/34 [==============================] - 1s 15ms/step - loss: 0.0245
Epoch 40/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0246
Epoch 41/10000
34/34 [==============================] - 0s 13ms/step - loss: 0.0245
Epoch 4

34/34 [==============================] - 1s 15ms/step - loss: 0.0245
Epoch 52/10000
34/34 [==============================] - 1s 17ms/step - loss: 0.0246
Epoch 53/10000
34/34 [==============================] - 1s 16ms/step - loss: 0.0245
Epoch 54/10000
34/34 [==============================] - 1s 17ms/step - loss: 0.0246
Epoch 55/10000
34/34 [==============================] - 1s 15ms/step - loss: 0.0249
Epoch 56/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 57/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 58/10000
34/34 [==============================] - 0s 13ms/step - loss: 0.0244
Epoch 59/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 60/10000
34/34 [==============================] - 0s 13ms/step - loss: 0.0246
Epoch 61/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0244
Epoch 62/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 6

34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 73/10000
34/34 [==============================] - 0s 13ms/step - loss: 0.0245
Epoch 74/10000
34/34 [==============================] - 0s 12ms/step - loss: 0.0245
Epoch 75/10000
34/34 [==============================] - 0s 13ms/step - loss: 0.0246
Epoch 76/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 77/10000
34/34 [==============================] - 0s 13ms/step - loss: 0.0246
Epoch 78/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0244
Epoch 79/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0246
Epoch 80/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 81/10000
34/34 [==============================] - 0s 13ms/step - loss: 0.0246
Epoch 82/10000
34/34 [==============================] - 0s 13ms/step - loss: 0.0247
Epoch 83/10000
34/34 [==============================] - 0s 12ms/step - loss: 0.0246
Epoch 8

...

34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 178/10000
34/34 [==============================] - 0s 13ms/step - loss: 0.0245
Epoch 179/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0246
Epoch 180/10000
34/34 [==============================] - 0s 13ms/step - loss: 0.0246
Epoch 181/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 182/10000
34/34 [==============================] - 0s 13ms/step - loss: 0.0246
Epoch 183/10000
34/34 [==============================] - 0s 12ms/step - loss: 0.0245
Epoch 184/10000
34/34 [==============================] - 0s 13ms/step - loss: 0.0246
Epoch 185/10000
34/34 [==============================] - 0s 13ms/step - loss: 0.0245
Epoch 186/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 187/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 188/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0

34/34 [==============================] - 1s 15ms/step - loss: 0.0245
Epoch 199/10000
34/34 [==============================] - 0s 15ms/step - loss: 0.0245
Epoch 200/10000
34/34 [==============================] - 1s 17ms/step - loss: 0.0245
Epoch 201/10000
34/34 [==============================] - 1s 16ms/step - loss: 0.0246
Epoch 202/10000
34/34 [==============================] - 1s 15ms/step - loss: 0.0247
Epoch 203/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 204/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0246
Epoch 205/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 206/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0246
Epoch 207/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 208/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0246
Epoch 209/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0

34/34 [==============================] - 0s 14ms/step - loss: 0.0247
Epoch 240/10000
34/34 [==============================] - 0s 13ms/step - loss: 0.0246
Epoch 241/10000
34/34 [==============================] - 0s 15ms/step - loss: 0.0245
Epoch 242/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 243/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 244/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 245/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 246/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 247/10000
34/34 [==============================] - 0s 13ms/step - loss: 0.0245
Epoch 248/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0246
Epoch 249/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0246
Epoch 250/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0

34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 261/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 262/10000
34/34 [==============================] - 1s 15ms/step - loss: 0.0244
Epoch 263/10000
34/34 [==============================] - 0s 15ms/step - loss: 0.0244
Epoch 264/10000
34/34 [==============================] - 1s 18ms/step - loss: 0.0246
Epoch 265/10000
34/34 [==============================] - 1s 18ms/step - loss: 0.0244
Epoch 266/10000
34/34 [==============================] - 0s 15ms/step - loss: 0.0244
Epoch 267/10000
34/34 [==============================] - 1s 15ms/step - loss: 0.0246
Epoch 268/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 269/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0245
Epoch 270/10000
34/34 [==============================] - 0s 15ms/step - loss: 0.0246
Epoch 271/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0

34/34 [==============================] - 0s 13ms/step - loss: 0.0246
Epoch 282/10000
34/34 [==============================] - 0s 13ms/step - loss: 0.0246
Epoch 283/10000
34/34 [==============================] - 0s 14ms/step - loss: 0.0246
Epoch 284/10000
33/34 [============================>.] - ETA: 0s - loss: 0.0238

In [ ]:
# predict
# prediction = cp.predict(testX, testY, 0, lap = lapacian) 
# prediction = np.rint(prediction).tolist()
# cp.polygonUpdate(prediction)

# Visualization
# # Point map
# cp.crimePointMap(np.random.choice(range(testX.shape[0]), size=1)[0]) 
# # Polygon map
# cp.crimePolygonMap() 
# Point and polygon map
# fig = cp.crimePolygonPointMap()  
# # get static
# cp.polygonUpdate(cp.getRule0Plan(7).tolist()) 
# cp.crimePolygonMap() 

# Actually predict (only X no Y) with visualization
# the second latestX is just for fill the position, the testY does not exist
prediction = cp.predict(X4Predict, np.empty(X4Predict.shape), 0, lap = lapacian) 
cp.polygonUpdate(prediction) 
cp.crimePolygonMap() 

In [ ]:
# grid search for hyper-parameters
def gridSearch():
    tf.random.set_seed(10)
    # lr_space = [0.000005, 0.0000075, 0.000010, 0.0000125, 0.000015, 0.0000175, 0.000020]
    # alpha_space = [3, 5, 7, 9]
    # lowerMultiplier_space = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    lr_space = [0.000020] 
    alpha_space = [5] 
    lowerMultiplier_space = [0.9]
    gridSearch_recorder = []
    for lr in lr_space:
        for alpha in alpha_space:
            for lowerM in lowerMultiplier_space:
                global trainedModel
                trainedModel = cp.train(trainX, trainY, valX, valY, lap_train = lapacian_train, lap_val = lapacian_val, \
                                        batch_size = 6, epochs = 10000, learning_rate = lr, patienceReduceLR = 5, \
                                       alpha_roundApproximation = alpha, lowerMultiplier = lowerM) 
                sta_rmse, dy_rmse, sta_mae, dy_mae, sta_r2, dy_r2 = cp.compareError(cp.getRule0Plan(7), testX, testY)
                sta_js, dy_js = cp.compareJSdivergence(cp.getRule0Plan(7), testX, testY)
                record = {'lr': lr, 'alpha': alpha, 'lowerM': lowerM,
                          'dy_js': dy_js, 'dy_rmse': dy_rmse, 'dy_mae': dy_mae, 'dy_r2': dy_r2,
                          'sta_js': sta_js, 'sta_rmse': sta_rmse, 'sta_mae': sta_mae, 'sta_r2': sta_r2,
                         }
                print(record)
                gridSearch_recorder.append(record)

# Monte Carlo simulation

In [ ]:
import networkx as nx
import shapely.ops as ops

def getPopulationonRoads(population, polygon):
    populationLabelled = population.sjoin(polygon, predicate = 'within')
    populationLabelled = populationLabelled.loc[:, ['#people', 'OBJECTID', 'geometry']]
    polygonLabelled = populationLabelled.dissolve(by = 'OBJECTID', aggfunc = 'sum')
    return polygonLabelled 

def connectLine(pf_road, manage_disconnection = False):
    # USE: Connect the Linestring objects in each MultiLineString objects
    #      Some MultiLineString objects contains LineString objects that are not connected to each other
    #      This function can fix the connecttion problem
    #INPUT: GeoPandas object read from shapefile
    #OUTPUT: Edited GeoPandas object
    idd = 0
    for gg in pf_road.geometry:
        # use the API to merge
        if str(type(gg)) == "<class 'shapely.geometry.multilinestring.MultiLineString'>":
            gg = ops.linemerge(gg)  
        pf_road.geometry[idd] = gg  
        # manage the slighted disconnected ones
        if manage_disconnection == True:
            if str(type(gg)) == "<class 'shapely.geometry.multilinestring.MultiLineString'>":
                gg_decom = [tuple(x.coords) for x in list(gg)]
                gg_c = -1
                for gg_p in gg:
                    gg_c += 1
                    if gg_p.boundary.is_empty:
                        del gg_decom[gg_c]

                gg_com = MultiLineString(gg_decom) 
                for gg1, gg2 in zip(gg_com, gg_com[1:]):
                    newline = LineString([Point(gg1.boundary[1].x, gg1.boundary[1].y),Point(gg2.boundary[0].x, gg2.boundary[0].y)])
                    gg_decom = gg_decom + [tuple(newline.coords)]

                gg = MultiLineString(gg_decom)
                gg = ops.linemerge(gg)
                pf_road.geometry[idd] = gg
        idd += 1
    return pf_road

def roads2Network(pf_input, roadWithPop, buffer = 0):
    #USE: Transform geological road network to topological network
    #INPUT: pf_input: GeoPandas object read from shapefile
    #       buffer: distances below the buffer lead to a connection between lines
    #OUTPUT: NetworkX graph object
    G = nx.Graph()
    i = 1
    for roadLength in pf_input.to_crs("EPSG:3035").length:
        G.add_node(i, length = roadLength)
        # set population
        try:
            G.nodes[i]['pop'] = roadWithPop['#people'].loc[i]
        except: # in case there is no people on this road/polygon
            G.nodes[i]['pop'] = 0
        i += 1
    for obid_1, road_1, length_1 in zip(pf_input.OBJECTID, pf_input.geometry, pf_input.to_crs("EPSG:3035").length):
        for obid_2, road_2, length_2 in zip(pf_input.OBJECTID, pf_input.geometry, pf_input.to_crs("EPSG:3035").length):
            if (obid_1 != obid_2) and road_1.distance(road_2) <= buffer:
                G.add_edge(obid_1, obid_2, weight = (length_1 + length_2) / 2)
            if (obid_1 != obid_2) and (road_1.distance(road_2) < buffer) and (road_1.distance(road_2) > 0):
                print(obid_1, obid_2)
    return G

def degreeCheck(graph):
    #USE: Check the number of lines connecting to nodes
    #INPUT: Networkx graph object
    #OUTPUT: Node degree report
    count = [0, 0, 0]
    for i in range(graph.number_of_nodes()):
        print('Node number: ', i + 1, '; Node degree: ', graph.degree[i + 1])
        if graph.degree[i + 1] < 8:
            count[0] += 1
        elif graph.degree[i + 1] == 8:
            count[1] += 1
        else:
            count[2] += 1
    print('Number of nodes with degree below 8: ', count[0])
    print('Number of nodes with degree equals 8: ', count[1])
    print('Number of nodes with degree above 8: ', count[2])   

    
roadData = gpd.read_file("./data/MajorRoads.shp").set_crs("EPSG:2240").to_crs("EPSG:4326")
roadData.insert(0, 'OBJECTID', range(1, 1 + len(roadData)))

roadData['length'] = (roadData.to_crs("EPSG:3035"))['geometry'].length # Add road length
roadData['midpoint'] = (roadData.to_crs("EPSG:3035"))['geometry'].centroid.to_crs("EPSG:4326") # Add midpoints to major road geoDataframe

population = gpd.read_file("./data/shapefile_populationDist/populationDist.shp")
polygon = gpd.read_file("./data/ThiessenPolygons.shp").set_crs("EPSG:2240").to_crs("EPSG:4326")
roadWithPop = getPopulationonRoads(population, polygon)

roadData = connectLine(roadData)
G = roads2Network(roadData, roadWithPop)

In [ ]:
# sim.degreeCheck(sim.G)
# plt.figure(figsize=(10,10))
# nx.draw_spring(sim.G, node_size=50, with_labels=True)

In [ ]:
import random

class simulation(crimePrediction):
    def __init__(self, origins, roadData, G, runNum, randomSeed, polygon = None, mapboxKey = ''):       
        random.seed(randomSeed) #set random seed
        self.runNum = runNum 
        self.origins = origins
        self.roadData = roadData
        self.startPoints = self.buildStartPoints(origins)
        self.G = G
        self.polygon = polygon        
        self.mapboxKey = mapboxKey      
        
    def changeRandomSeedTo(self, randomSeed):
        random.seed(randomSeed)
    
    def buildStartPoints(self, origins):
        start_points = []
        loc = 1
        for item in origins:
            if item != None and item >= 1:
                start_points.append(loc)
            loc += 1 
        return start_points
           
    ##########################################################################
    ############################ Simulation ##################################    
    def _vehicleRoute_fromPredictedLocations(self, node_num):
        #OUTPUT: list of nodes
        randomNode = random.choice(list(self.G.nodes())) # randomly get a node in the graph
        route = nx.shortest_path(self.G, weight = 'weight', source = node_num, target = randomNode)
        return route

    def _vehicleRoute_toPredictedLocations(self, node_num):
        #OUTPUT: list of nodes
        # randomly choose road according to population
        roads = [item[0] for item in list(self.G.nodes(data = 'pop'))]
        pops = [item[1] for item in list(self.G.nodes(data = 'pop'))]
        pops_dist = pops / sum(pops)
        randomNode = random.choices(roads, pops_dist)[0]
        # get route (in a reverse way, from pop dist to predicted crime incident locations)
        route = nx.shortest_path(self.G, weight = 'weight', source = randomNode, target = node_num)
        return route
    
    def _roadsSeperationByDirection(self, connectedRoads, buffer = 0, lost_warning = False):
        # USE: assign the connected roads to the two end points of the road for analysis
        # INPUT: dict, the connected roads for a road
        # OUTPUT: a dict
        
        # get the geometry of this road and the coordinates of this line
        get_line = self.roadData[self.roadData.OBJECTID == list(connectedRoads.keys())[0]].geometry.values[0]
        try:
            coords = get_line.coords
        except: # When lines are disconnected, data problem
            coords = [get_line[0].coords[0], get_line[-1].coords[-1]]
            if lost_warning == True:
                print("Remedial measures are taken to extract end points from road segments. Please note there are problems (i.e., lines are not connected in the same group) in the data, although they are solved for this time.")
        
        # assign the connected roads to the two end points of the road in analysis
        c1 = []
        c2 = []
        for road in list(connectedRoads.values())[0]:
            connectedRoadLine = self.roadData[self.roadData.OBJECTID == road].geometry.values[0]
            if connectedRoadLine.distance(Point(coords[0])) <= buffer: # if the connected raod is closer to the cor 0
                c1.append(road)
            elif connectedRoadLine.distance(Point(coords[-1])) <= buffer: # if the connected raod is closer to the cor 1
                c2.append(road)
            else:
                if lost_warning == True:
                    print('Road', road, 'is not attached to any vetice of the target road', list(connectedRoads.keys())[0], '.')
        return {'v1': c1, 'v2': c2}

    def _placementWeight(self, recordsByRoads, roadClustersByDirection):
        # USE: Calculate the weight of each placement using simulation records and road cluster information
        # INPUT: recordsByRoads - all simulation record grouped by roads
        #        roadClustersByDirection - road cluster information for each road    

        placement_weight = []
        for i in range(len(recordsByRoads)):
            record = recordsByRoads[i + 1] 
            roadset_clustered = roadClustersByDirection[i + 1]
            
            # Calculate average detection number for each road pair 
            # (# of vehicles detected among all iteration / total # of iteration)
            roads_v1 = roadset_clustered['v1']
            roads_v2 = roadset_clustered['v2']
            detection_num_v1_temp = [[i['iteration'], i['origin'], i['sequence on origin']] \
                                     for i in record if (i['step1'] in roads_v1 or i['step2'] in roads_v2)] # start is on v1 or end is on v2
            detection_num_v2_temp = [[i['iteration'], i['origin'], i['sequence on origin']] \
                                     for i in record if (i['step1'] in roads_v2 or i['step2'] in roads_v1)] # start is on v2 or end is on v1
            # multiplying by 2 as records are counted twice for each road (in and out)
            ave_detection_num_v1 = len(detection_num_v1_temp) / (2 * self.runNum) 
            ave_detection_num_v2 = len(detection_num_v2_temp) / (2 * self.runNum) 
            
            # Tidy up the data
            out1 = {'Road': i + 1, 'Inflow roads':roads_v1, 'Outflow roads': roads_v2, 'Number of detected vehicles': ave_detection_num_v1}
            out2 = {'Road': i + 1, 'Inflow roads': roads_v2, 'Outflow roads': roads_v1, 'Number of detected vehicles': ave_detection_num_v2}
                  
            placement_weight.append(out1)
            placement_weight.append(out2)
        placement_weight_sorted = sorted(placement_weight, reverse = True, key = lambda x: x['Number of detected vehicles'])
        return placement_weight_sorted

    def getRoadSeperation(self):
        roadClustersByDirection = {k + 1: None for k in range(self.G.number_of_nodes())}
        for road in range(self.G.number_of_nodes()):
            roadsNeighbor = list(nx.neighbors(self.G, road + 1))
            roadsConnected = {road + 1: roadsNeighbor}
            roadset_clustered = self._roadsSeperationByDirection(roadsConnected) 
            roadClustersByDirection[road + 1] = roadset_clustered
        return roadClustersByDirection
        
    def runInitialization(self):
        # USE: Calculate the weight of each "placement"
        #     Each "placement" is the combination of a road segment and a direction,
        #     representing the placement of a camera
        # USE: run the simulation and record all the data for further analysis
        # OUTPUT: recordsByRoads - simulation record on each roads
        #         recordsByIterations - simulation record by each simulation iteration, each iteration contains routes of each vehicle
        #         roadClustersByDirection - road cluster information for each road

        # Record all information in the simulation of each road segment / of each iteration
        recordsByRoads = {k + 1: [] for k in range(self.roadData.shape[0])} # Output: the showing up of vehicel on each road 
        recordsByIterations = {k + 1: None for k in range(self.runNum)} # Output: route record of all vehicles of all iterations
        for iter_0 in range(self.runNum):
            routesOnOrigins = {k: None for k in self.startPoints}
            for origin in self.startPoints: 
                routesOfVehicles = {k + 1: None for k in range(int(self.origins[origin - 1]))}
                for vehicle_0 in range(int(self.origins[origin - 1])):
                    route = self._vehicleRoute_toPredictedLocations(origin)
                    routesOfVehicles[vehicle_0 + 1] = route
                    for start, end in zip(route[:-1], route[1:]):
                        detection = {'iteration': iter_0 + 1, 'origin': origin, 'sequence on origin': vehicle_0 + 1, 'step1': start, 'step2': end}
                        recordsByRoads[start].append(detection)
                        recordsByRoads[end].append(detection)
                routesOnOrigins[origin] = routesOfVehicles
            recordsByIterations[iter_0 + 1] = routesOnOrigins    
        
        # Seperating the connected roads to two direction for each road
        roadClustersByDirection = self.getRoadSeperation()

        return recordsByRoads, recordsByIterations, roadClustersByDirection
    
    def _delete_judge (self, delete_list, test):
        iteration = test[0]
        origin = test[1]
        vehicle = test[2]
        captured_vehicle = [[i['origin'], i['vehicle_num']] for i in delete_list if i['iteration_num'] == iteration]
        if [origin, vehicle] in captured_vehicle:
            return True
        else:
            return False
        
    def placementElemination(self, placement_weight_sorted, recordsByRoads, recordsByIterations, withEffectRatio = False):
        # USE: Eliminate the contribution of a placed camera
        #      Return a data without the contribution for calculating the best placement in the rest
        # OUTPUT: updated version of recordsByRoads

        # Get target movements from the placement
        inflow = [[i, placement_weight_sorted[0]['Road']] for i in placement_weight_sorted[0]['Inflow roads']]
        outflow = [[placement_weight_sorted[0]['Road'], i] for i in placement_weight_sorted[0]['Outflow roads']]
        valid_flows = inflow + outflow
        
        # Check if any of the movements exist in recordsByIterations
        # If so, get the index of simulation iteration and the index of suspect vehicle
        # Input: valid_flows
        # Output: delete_list
        delete_list = []
        for iteration_index_0 in range(len(recordsByIterations)):
            reco_iteration = recordsByIterations[iteration_index_0 + 1]
            for origin in list(reco_iteration.keys()):
                for vehicle, get_route in zip(list(reco_iteration[origin].keys()), list(reco_iteration[origin].values())):
                    # judge if the route contains the valid flows indicated by the placement plan
                    for flow in valid_flows:  
                        if flow[0] in get_route:
                            next_step_index = [index for index, x in enumerate(get_route) if x == flow[0]]
                            next_step = []
                            for k in next_step_index:
                                if k == len(get_route) - 1:
                                    continue
                                else:
                                    next_step.append(get_route[k + 1])
                            if flow[1] in next_step:
                                if withEffectRatio == False:
                                    delete_list.append({'iteration_num': iteration_index_0 + 1, 'origin': origin, 'vehicle_num': vehicle})
                                
                                # if consider pre-located cameras
                                elif withEffectRatio == True:
                                    if random.random() <= placement_weight_sorted[0]['EffectRatio']:
                                        delete_list.append({'iteration_num': iteration_index_0 + 1, 'origin': origin, 'vehicle_num': vehicle})
                                        
        # a vehicel from a origin in a iteration may travel through multiple valid flows
        delete_list_short = []
        [delete_list_short.append(x) for x in delete_list if x not in delete_list_short]
        
        global delete_list_short_length_ave
        delete_list_short_length_ave = len(delete_list_short) / self.runNum
        
        # According to the indexes obtained, delete corresponding items from recordsByIterations
        for delete in delete_list_short:
            del recordsByIterations[delete['iteration_num']][delete['origin']][delete['vehicle_num']]
        # According to the indexes obtained, delete corresponding items from recordsByRoads
        updated_recordsByRoads = {k + 1: [] for k in range(self.roadData.shape[0])}
        for road_index_0 in range(len(recordsByRoads)):
            updated_road_reco = [i for i in recordsByRoads[road_index_0 + 1] \
                                 if self._delete_judge(delete_list_short, [i['iteration'], i['origin'], i['sequence on origin']]) == False]
            updated_recordsByRoads[road_index_0 + 1] = updated_road_reco
        return updated_recordsByRoads, recordsByIterations

    def placeMultipleCamera(self, required_camera, preLocated_cameras = None):
        # preLocated_cameras: a list of dict, each dict has "road" and "effect_ratio" keys
        
        # Initialization, run simualtion, and get the first placement of camera
        layout = {k + 1: [] for k in range(required_camera)}
        recordsByRoads, recordsByIterations, roadset_clustered_list = self.runInitialization()
        
        # Consider the existing cameras (virtual cam due to residule effect or fixed one)
        if preLocated_cameras != None: 
            global layout_forPreLocated
            layout_forPreLocated = {k + 1: [] for k in range(len(preLocated_cameras))}
            for k in range(len(preLocated_cameras)):
                preLocated_camera = [preLocated_cameras[k]]
                recordsByRoads, recordsByIterations = \
                    self.placementElemination(preLocated_camera, recordsByRoads, recordsByIterations, withEffectRatio = True)
                layout_forPreLocated[k + 1] = {'Road': preLocated_cameras[k]['Road'], 
                                               'Inflow roads': preLocated_cameras[k]['Inflow roads'], 
                                               'Outflow roads': preLocated_cameras[k]['Outflow roads'], 
                                               'Number of detected vehicles': delete_list_short_length_ave}
        
        # First cam
        placement_weight_sorted = self._placementWeight(recordsByRoads, roadset_clustered_list)
        layout[1] = placement_weight_sorted[0]
        
        # Generating multiple camera placement
        updated_recordsByRoads = recordsByRoads
        updated_recordsByIterations = recordsByIterations
        for i in range(required_camera - 1):
            # Eliminating contribution (of last camera)
            updated_recordsByRoads, updated_recordsByIterations = \
                self.placementElemination(placement_weight_sorted, updated_recordsByRoads, updated_recordsByIterations)
            # Recalculate weights
            placement_weight_sorted = self._placementWeight(updated_recordsByRoads, roadset_clustered_list)
            # Add placement
            layout[i + 2] = placement_weight_sorted[0]
        return layout
       

    ################################################################################
    ########################### Visulization #######################################
    def _get_orientation(self, cor_1, cor_2):
        # INPUT: two lists of [lat,lon]
        origin_y = cor_1[0]
        origin_x = cor_1[1]
        destination_y = cor_2[0]
        destination_x = cor_2[1]    
        deltaX = destination_x - origin_x
        deltaY = destination_y - origin_y
        degrees_temp = math.atan2(deltaX, deltaY)/ math.pi*180
        if degrees_temp < 0:
            degrees_final = 360 + degrees_temp
        else:
            degrees_final = degrees_temp
        compass_brackets = ["North", "Northeast", "East", "Southeast", "South", "Southwest", "West", "Northwest", "North"]
        compass_lookup = round(degrees_final / 45)
        return compass_brackets[compass_lookup], degrees_final
    
    def placementVisualizationMapbox(self, layout, timeRange_text = None, show_raw_data = False):
        # Draw polygons
#         geo_df_select = self.polygon.loc[self.polygon.Prediction > 0] # only draw polygon with predicted crime
        geo_df_select = self.polygon
        geo_df_select['Prediction'] = geo_df_select['Prediction'].astype(str)
        fig = px.choropleth_mapbox(geo_df_select, 
                                   geojson = geo_df_select.geometry,
                                   locations = geo_df_select.index, 
                                   color = "Prediction", 
                                   color_discrete_sequence = px.colors.sequential.Reds,
                                   center = {"lat": 32.606827, "lon": -83.660198},
                                   zoom = 11.8, 
                                   opacity = 0.6, 
                                   hover_data = ['Prediction'],
                                  )  
        fig.update_traces(hovertemplate = "<b>Number of Predicted Crime:</b> %{customdata} <extra></extra>")            
        # Calculate orientation
        road_list = [item['Road'] for item in layout.values()]
        road_out_list = [item['Outflow roads'] for item in layout.values()]
        direction_dict = dict.fromkeys(road_list, None)
        for origin, destination in zip(road_list, road_out_list):
            point1 = self.roadData.loc[self.roadData['OBJECTID'] == origin].midpoint
            road1 = self.roadData.loc[self.roadData['OBJECTID'] == origin].geometry.reset_index()
            road2 = self.roadData.loc[self.roadData['OBJECTID'] == destination[0]].geometry.reset_index()
            point2 = road1.intersection(road2)
            cor1 = [point1.y.values[0],point1.x.values[0]]
            cor2 = [point2.y.values[0],point2.x.values[0]]
            orientation, _ = self._get_orientation(cor1, cor2)
            if direction_dict[origin] == None:
                direction_dict[origin] = orientation
            else:
                direction_dict[origin] = direction_dict[origin] + ', ' + orientation
        # Prepare df
        points_selected = self.roadData.loc[self.roadData['OBJECTID'].isin(road_list)].copy()
        points_selected_index = points_selected.OBJECTID.tolist()
        direction_list = [direction_dict[road] for road in points_selected_index]
        points_selected['direction'] = direction_list
        # Draw scatter plot
        fig2 = px.scatter_mapbox(points_selected, 
                                 lat = points_selected.midpoint.y, 
                                 lon = points_selected.midpoint.x,
                                 opacity = 0.7, 
                                 hover_data = [points_selected.midpoint.y, points_selected.midpoint.x, points_selected.direction])
        fig2.update_traces(marker = dict(size = 10, color = '#D30000'),
                           hovertemplate = "<b>Lat:</b> %{customdata[0]}<br> <b>Lon:</b> %{customdata[1]}<br> <b>Direction:</b> %{customdata[2]} <extra></extra> ")
        fig.add_trace(fig2.data[0])
        # Layout
        fig.update_layout(mapbox_style = "light",
                          mapbox_accesstoken = self.mapboxKey,
                          title_text = 'LPR Placement Plan and Crime Pattern Prediction' + '  (' + timeRange_text + ')',
                          margin = {"r":30,"l":30,"b":60}, 
                          autosize = False, width = 800, height = 600,
                          annotations = [dict(text = "Note: LPRs will maintain same theoretical effectiveness placed within the road segment range. <br>Network Dynamics Lab: http://ndl.gatech.edu/",
                                              xref = "paper", yref = "paper",
                                              x = 0, y = -0.08, 
                                              showarrow = False,
                                              align = 'left', )],)
        # Combine raw crime location
        if show_raw_data == True:
            fig_raw = self.crimePointMap(self.testWeek)
            fig.add_trace(fig_raw.data[0]) 
        return fig


In [ ]:
cp.pf.Prediction = cp.pf.Prediction.astype(int)
sim = simulation(prediction, roadData, G, 250, 10, polygon = cp.pf, \
                 mapboxKey = 'pk.eyJ1IjoicHhpeW9oIiwiYSI6ImNsMGoxa3h1bzA4ZHQzaW41NWd6dm16am0ifQ.QywfLC6Ut-EhSZLt7nirqQ')

In [ ]:
# layout_withoutResidual = sim.placeMultipleCamera(required_camera = 10)
layout_withResidual = sim.placeMultipleCamera(required_camera = 10, preLocated_cameras = 
                                 [{'Road': 106, 'Inflow roads': [56, 208], 'Outflow roads': [50, 75, 107], 'EffectRatio': 0.3}, 
                                  {'Road': 76, 'Inflow roads': [125, 201], 'Outflow roads': [203, 266], 'EffectRatio': 0.5}, 
                                  {'Road': 38, 'Inflow roads': [41, 249, 258], 'Outflow roads': [262, 274], 'EffectRatio': 0.4}])
print(layout_withResidual)

In [ ]:
sim.placementVisualizationMapbox(layout, timeRange_text = '30th Jan. to 5th Feb.').write_html('cameraPlacement_withResidual.html')

In [ ]:
print(layout_withoutResidual)
print(layout_withResidual)
print(layout_forPreLocated)

layout_withoutResidual_list = [layout_withoutResidual[k + 1]['Number of detected vehicles'] for k in range(len(layout_withoutResidual))]
layout_withoutResidual_list = [0, 0, 0] + layout_withoutResidual_list
accumulate_layout_withoutResidual_list = [sum(layout_withoutResidual_list[:i + 1]) for i in range(len(layout_withoutResidual_list))]

layout_withResidual_list = [layout_withResidual[k + 1]['Number of detected vehicles'] for k in range(len(layout_withResidual))]
layout_withResidual_list = [0, 0, 0] + layout_withResidual_list
accumulate_layout_withResidual_list = [sum(layout_withResidual_list[:i + 1]) for i in range(len(layout_withResidual_list))]

layout_withResidual_withVirtualOnes_list = [layout_withResidual[k + 1]['Number of detected vehicles'] for k in range(len(layout_withResidual))]
layout_withResidual_withVirtualOnes_list = [layout_forPreLocated[k + 1]['Number of detected vehicles'] for k in range(len(layout_forPreLocated))] \
    + layout_withResidual_withVirtualOnes_list
accumulate_layout_withResidual_withVirtualOnes_list = [sum(layout_withResidual_withVirtualOnes_list[:i + 1]) \
                                                       for i in range(len(layout_withResidual_withVirtualOnes_list))]

fig = go.Figure()
camNum = range(len(accumulate_layout_withResidual_list))
fig.add_trace(go.Scatter(x = [item + 1 for item in camNum], y = accumulate_layout_withoutResidual_list, fill = 'tozeroy', name = 'Real effect')) 
fig.add_trace(go.Scatter(x = [item + 1 for item in camNum], y = accumulate_layout_withResidual_list, fill = 'tozeroy', name = 'Real effect considering residual effect')) 
fig.add_trace(go.Scatter(x = [item + 1 for item in camNum], y = accumulate_layout_withResidual_withVirtualOnes_list, fill = 'tonexty', name = 'Real effect + Residual effect')) 
fig.update_layout(xaxis_title="Index of cameras", yaxis_title="# of vehicles captured (124 in total)")
fig.show()

# Validation

In [ ]:
import random
import plotly.graph_objects as go

def simWithRawData(weekInTestData, runTimes, routeLength, randomSeed):
    crimeMap = testY[weekInTestData] # get the real crime places
    simulator = simulation(crimeMap, roadData, G, runTimes, routeLength, randomSeed) # import simulator and run
    recordsByRoads, recordsByIterations, roadClustersByDirection = simulator.runInitialization()
    return recordsByRoads, recordsByIterations

def calcCapturedVehicleNum(cameraPlacement, recordsByRoads, recordsByIterations, runTimes, routeLength, randomSeed = 0):
    # INPUT: cameraPlacement format: {1: {'Road': 119, 'Inflow roads': [242, 241], 'Outflow roads': [198, 273, 275]}, {2: ...}
    #        recordsByRoads format: {1: [[iterationNum, origin, vehicleNum, InflowRoadNum, 1], [3, 57, 1, 280, 1], ...], 2:[  ...}
    simulator = simulation([], roadData, None, None, routeLength, randomSeed)
    CapturedVehicleNum = 0
    updated_recordsByRoads, updated_recordsByIterations = recordsByRoads, recordsByIterations
    
    for k in range(len(cameraPlacement)):
        k = k + 1
        # info of camera
        road = cameraPlacement[k]['Road']
        inflow = cameraPlacement[k]['Inflow roads']
        outflow = cameraPlacement[k]['Outflow roads']
        validFlows = [[int(j), int(road)] for j in inflow] + [[road, i] for i in outflow]
        # info of flow
        records = updated_recordsByRoads[road]
        for record in records: # record is a dict [iterationNum, origin, vehicleNum, inflow, outflow]
            flow = [int(record['step1']), int(record['step2'])]
            if flow in validFlows:
                CapturedVehicleNum += 1
        # update recordsByRoads, eliminating the effect of the last camera
        updated_recordsByRoads, updated_recordsByIterations = simulator.placementElemination( \
            [cameraPlacement[k]], updated_recordsByRoads, updated_recordsByIterations)
        
    ave_CapturedVehicleNum = CapturedVehicleNum / (2 * runTimes)            
    return ave_CapturedVehicleNum

def getBaseline1(N, roadClustersByDirection):
    # Baseline 1 : taking the average of crime numbers in polygons, getting top 5 locations, setting cameras in both directions
    # INPUT: N - int, even number
    #        roadClustersByDirection - {1: {'v1': [169, 180, 220], 'v2': [268, 280]}, ...
    if (N%2) != 0:
        raise Exception("N should be even number.")
    # get the number of top n roads
    crimeMapFull = dp.crimeRaster(df)
    topIndex = crimeMapFull.argsort().tolist()[0][-int(N / 2):]
    topIndex.reverse()
    topRoad = [i + 1 for i in topIndex]
    # assign inflow randomly and generate plan
    cameraPlacement = {k + 1: {'Road': None, 'Inflow roads': None, 'Outflow roads': None} for k in range(N)}
    for road, camera in zip(topRoad, range(int(N / 2))):
        inflow = roadClustersByDirection[road]['v1']
        outflow = roadClustersByDirection[road]['v2']
        cameraPlacement[camera + 1]['Road'] = road
        cameraPlacement[camera + 1]['Inflow roads'] = inflow
        cameraPlacement[camera + 1]['Outflow roads'] = outflow
    for road, camera in zip(topRoad, range(int(N / 2))):
        inflow = roadClustersByDirection[road]['v2']
        outflow = roadClustersByDirection[road]['v1']
        cameraPlacement[(N / 2) + camera + 1]['Road'] = road
        cameraPlacement[(N / 2) + camera + 1]['Inflow roads'] = inflow
        cameraPlacement[(N / 2) + camera + 1]['Outflow roads'] = outflow
    return cameraPlacement

def getBaseline2(N, roadClustersByDirection):
    # Baseline 2:  taking the average of crime numbers in polygons, getting top 10 locations, setting cameras with random directions
    # INPUT: N - int, the number of camera
    #        roadClustersByDirection - {1: {'v1': [169, 180, 220], 'v2': [268, 280]}, ...
    
    # get the number of top n roads
    crimeMapFull = dp.crimeRaster(df)
    topIndex = crimeMapFull.argsort().tolist()[0][-N:]
    topIndex.reverse()
    topRoad = [i + 1 for i in topIndex]
    # assign inflow randomly and generate plan
    cameraPlacement = {k + 1: {'Road': None, 'Inflow roads': None, 'Outflow roads': None} for k in range(N)}
    for road, camera in zip(topRoad, range(N)):
        ran = random.randint(0, 1)
        if ran == 0:
            inflow = roadClustersByDirection[road]['v1']
            outflow = roadClustersByDirection[road]['v2']
        elif ran == 1:
            inflow = roadClustersByDirection[road]['v2']
            outflow = roadClustersByDirection[road]['v1']
        cameraPlacement[camera + 1]['Road'] = road
        cameraPlacement[camera + 1]['Inflow roads'] = inflow
        cameraPlacement[camera + 1]['Outflow roads'] = outflow
    return cameraPlacement

def getBaseline3(N, runTimes, routeLength, randomSeed, dayIterval = 7):
    # Baseline 3:  using all the crime points (sample it to reduce computation time) as the starting points of the vehicles, 
    #              setting cameras with simulation
    # INPUT: N, int, the number of cameras
    # OUTPUT: camera placement plan
    
    # slice data from the dataframe and reduce the number of crime of each polygon with same scale
    dfs = df.iloc[0 : int(len(df) / 10)]
    weeks = int ((dfs.index[0] - dfs.index[-1]).days / dayIterval)
    crimeMapFull = dp.crimeRaster(dfs) / weeks
#     crimeMapFull[crimeMapFull < (10 / weeks)] = 0 # to cover 90% of the crimes
    crimeMapFull = np.rint(crimeMapFull)
    # import simulator and run
    simulator = simulation(crimeMapFull.tolist()[0], roadData, G, runTimes, routeLength, randomSeed)
    cameraPlacement = simulator.placeMultipleCamera(required_camera = N)
    return cameraPlacement

def getMyPlan(weekInTestData, cameraNum, CrimeThreshold, runTime = 250, routeLength = 15, randomSeed = random.randint(0, 100)):
    # generate plan using the proposed method
    predictY = cp.predict(testX, testY, weekInTestData, lap = lapacian)
    sim = simulation(predictY, roadData, G, runTime, routeLength, randomSeed)
    myPlacement = sim.placeMultipleCamera(required_camera = cameraNum)
    return myPlacement

def comparePlans(cameraNum, CrimeThreshold, runTime = 250, routeLength = 15):
    
    base1 = []
    base2 = []
    base3 = []
    my = []
    
    roadClustersByDirection = simulation([], roadData, G, None, None, None).getRoadSeperation()
    cameraPlacement1 = getBaseline1(cameraNum, roadClustersByDirection)
    print('Baseline1 plan is generated.')
    cameraPlacement2 = getBaseline2(cameraNum, roadClustersByDirection)
    print('Baseline2 plan is generated.')
     
#     for week in range(testX.shape[0]): 
    for week in range(10): 
#     for week in [3, 4]: 
        randomSeed = week + 23
        recordsByRoads, recordsByIterations = simWithRawData(week, runTime, routeLength, randomSeed)
        # baseline1
        base1.append(calcCapturedVehicleNum(cameraPlacement1, recordsByRoads, recordsByIterations, runTime, routeLength))
        print('Week ', week, ' baseline1 is evaluated.')
        # baseline2
        base2.append(calcCapturedVehicleNum(cameraPlacement2, recordsByRoads, recordsByIterations, runTime, routeLength))
        print('Week ', week, ' baseline2 is evaluated.')
        # baseline3
        cameraPlacement3 = getBaseline3(cameraNum, runTime, routeLength, randomSeed)
        print(cameraPlacement3)
        print('Baseline3 plan is generated.')        
        base3.append(calcCapturedVehicleNum(cameraPlacement3, recordsByRoads, recordsByIterations, runTime, routeLength))
        print('Week ', week, ' baseline3 is evaluated.')
        # optimized plan
        myPlacement = getMyPlan(week, cameraNum, CrimeThreshold, randomSeed = randomSeed)
        print(myPlacement)
        print('Week ', week, ' my plan is generated.')
        my.append(calcCapturedVehicleNum(myPlacement, recordsByRoads, recordsByIterations, runTime, routeLength))
        print('Week ', week, ' my plan is evaluated.')
    return base1, base2, base3, my

def visValidation(base1 = None, base2 = None, base3 = None, my = None):
    fig = go.Figure()
    fig.add_trace(go.Box(name = 'Baseline 1', showlegend = False, y = base1))
    fig.add_trace(go.Box(name = 'Baseline 2', showlegend = False, y = base2))
    fig.add_trace(go.Box(name = 'Baseline 3', showlegend = False, y = base3))
    fig.add_trace(go.Box(name = 'Our plan', showlegend = False, y = my))
    return fig

In [ ]:
baseline1, baseline2, baseline3, my = comparePlans(10, 0.28, runTime = 150, routeLength = 25)
visValidation(baseline1, baseline2, baseline3, my).show()

# Utilities

In [ ]:
# get adjancency matrix of the roads and polygons
import networkx as nx
road_graph = sim.G.to_undirected()
A = nx.to_numpy_matrix(road_graph)
np.save('./data/adjMatrix', A)

In [ ]:
# draw the change of camera effectiveness due to considering residual effect
import plotly.express as px
df = px.data.gapminder()
fig = px.area(df, x="year", y="pop", color="continent", line_group="country")
fig.show()

In [ ]:
# Draw the roads in the city
import shapely
import plotly

roadData = gpd.read_file("./data/MajorRoads.shp").set_crs("EPSG:2240").to_crs("EPSG:4326")
roadData.insert(0, 'OBJECTID', range(1, 1 + len(roadData)))
roadData['midpoint'] = (roadData.to_crs("EPSG:3035"))['geometry'].centroid.to_crs("EPSG:4326") # Add midpoints to major road geoDataframe
# roadData = connectLine(roadData)

lats = []
lons = []
names = []

for feature, name in zip(roadData.geometry, roadData.OBJECTID):
    if isinstance(feature, shapely.geometry.linestring.LineString):
        linestrings = [feature]
    elif isinstance(feature, shapely.geometry.multilinestring.MultiLineString):
        linestrings = feature.geoms
    else:
        continue
    for linestring in linestrings:
        x, y = linestring.xy
        lats = np.append(lats, y)
        lons = np.append(lons, x)
        names = np.append(names, [name]*len(y))
        lats = np.append(lats, None)
        lons = np.append(lons, None)
        names = np.append(names, None)
        
mapboxKey = 'pk.eyJ1IjoicHhpeW9oIiwiYSI6ImNsMGoxa3h1bzA4ZHQzaW41NWd6dm16am0ifQ.QywfLC6Ut-EhSZLt7nirqQ'
plotly.express.set_mapbox_access_token(mapboxKey)
fig = px.line_mapbox(lat = lats, lon=lons, hover_name = names,
                     center = {"lat": 32.606827, "lon": -83.660198},
                     color = [1] * len(lats),
                     color_discrete_sequence = ['#F77257'],
                     zoom = 11.7,
                     mapbox_style="light"
                    )
# fig.show()